# MudraLingua ISL Landmark Training Notebook\nCNN-LSTM pipeline for ISL letters, numbers, and words using MediaPipe hand landmarks (126 features per frame).

In [ ]:
!pip install -q mediapipe opencv-python kaggle tensorflow scikit-learn pandas numpy

In [ ]:
import os, cv2, json, math, random\nimport numpy as np\nimport pandas as pd\nfrom pathlib import Path\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import LabelEncoder\nimport tensorflow as tf\nfrom tensorflow.keras import layers, models, callbacks\nimport mediapipe as mp

## 1) Download Kaggle datasets

In [ ]:
# Ensure kaggle.json is uploaded in ~/.kaggle first.\n!kaggle datasets download -d soumyakushwaha/indian-sign-language-dataset -p data --unzip\n!kaggle datasets download -d arvindvinod/indian-sign-language-dataset-part-1 -p data --unzip\n!kaggle datasets download -d drblack00/isl-csltr-indian-sign-language-dataset -p data --unzip

## 2) MediaPipe feature extraction (21 x [x,y,z] for two hands => 126 dims/frame)

In [ ]:
mp_hands = mp.solutions.hands\nhands = mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5)\n\ndef frame_to_landmarks(frame):\n    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n    result = hands.process(rgb)\n    feat = np.zeros((2,21,3), dtype=np.float32)\n    if result.multi_hand_landmarks:\n      for h_idx, hand_lm in enumerate(result.multi_hand_landmarks[:2]):\n        for i, lm in enumerate(hand_lm.landmark):\n          feat[h_idx, i] = [lm.x, lm.y, lm.z]\n    return feat.reshape(-1)  # 126\n\ndef video_to_sequence(video_path, seq_len=32):\n    cap = cv2.VideoCapture(str(video_path))\n    frames = []\n    while cap.isOpened():\n      ok, frm = cap.read()\n      if not ok: break\n      frames.append(frame_to_landmarks(frm))\n    cap.release()\n    if len(frames) == 0:\n      return np.zeros((seq_len,126), dtype=np.float32)\n    idx = np.linspace(0, len(frames)-1, seq_len).astype(int)\n    return np.array([frames[i] for i in idx], dtype=np.float32)

In [ ]:
def build_dataset(data_root='data', seq_len=32):\n    X, y = [], []\n    for cls_dir in Path(data_root).glob('*'):\n      if not cls_dir.is_dir():\n        continue\n      label = cls_dir.name\n      for vid in cls_dir.rglob('*.mp4'):\n        X.append(video_to_sequence(vid, seq_len=seq_len))\n        y.append(label)\n    return np.array(X), np.array(y)\n\nX, y = build_dataset('data', seq_len=32)\nprint('Dataset:', X.shape, y.shape)

## 3) CNN-LSTM model

In [ ]:
le = LabelEncoder()\ny_enc = le.fit_transform(y)\nnum_classes = len(le.classes_)\ny_cat = tf.keras.utils.to_categorical(y_enc, num_classes)\nX_train, X_val, y_train, y_val = train_test_split(X, y_cat, test_size=0.2, stratify=y_enc, random_state=42)\n\ninp = layers.Input(shape=(32,126))\nx = layers.Reshape((32,126,1))(inp)\nx = layers.TimeDistributed(layers.Conv1D(64, 3, padding='same', activation='relu'))(x)\nx = layers.TimeDistributed(layers.MaxPool1D(2))(x)\nx = layers.TimeDistributed(layers.Flatten())(x)\nx = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)\nx = layers.Bidirectional(layers.LSTM(64))(x)\nx = layers.Dropout(0.3)(x)\nout = layers.Dense(num_classes, activation='softmax')(x)\nmodel = models.Model(inp, out)\nmodel.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])\nmodel.summary()

In [ ]:
cb = [\n  callbacks.ModelCheckpoint('checkpoints/best.keras', monitor='val_accuracy', save_best_only=True),\n  callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),\n]\nhist = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=60, batch_size=32, callbacks=cb)\nval_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)\nprint('Validation accuracy:', val_acc)

### Targets\n- Letters: >=96%\n- Words: >=88%

In [ ]:
model.export('exported/saved_model')\nwith open('exported/labels.json', 'w') as f:\n    json.dump(le.classes_.tolist(), f, indent=2)\nprint('Saved model and labels.')

## 4) Convert to TFLite

In [ ]:
!python scripts/convert_to_tflite.py --saved_model_dir exported/saved_model --output assets/models/isl_sequence_model.tflite